# Week 11 lab — platform-ops walkthrough

You own the platform for the Week 10 RAG service. This notebook wires the five components you
implement in `src/platformops/` into **one refresh loop**:

```
serve → signals → drift + decay + feedback → RefreshTrigger → rebuild → ReleaseGate → CanaryController → promote
```

Run `make test` first; implement the `TODO`s until it's green; then run this end to end.
Everything here is offline (synthetic trace stream) — `make live` adds a real-judge check.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / "solution"))   # notebook uses the reference impl
sys.path.insert(0, str(pathlib.Path.cwd() / "fixtures"))
from platformops import (SignalAggregator, DriftMonitor, RefreshTrigger, ReleaseGate,
                         CanaryController, EvalReport, categorical_psi)
from trace_helpers import make_stream, measure_factory
import numpy as np
print("components loaded")

## 1 — Signals: watch quality, not just latency

In [ ]:
week1 = make_stream(2000, seed=1, topic_mix=(0.7, 0.3))
week4 = make_stream(2000, seed=4, topic_mix=(0.3, 0.7))   # 'new_product' questions took over

def roll(stream, window=800):
    agg = SignalAggregator(window)
    for e in stream:
        agg.observe(e)
    return agg.signals()

s1, s4 = roll(week1), roll(week4)
print("week 1:", s1)
print("week 4:", s4)
print(f"\ngrounded_rate {s1.grounded_rate} -> {s4.grounded_rate};  "
      f"error_rate barely moved ({s1.error_rate} -> {s4.error_rate})")

## 2 — Drift: quantify what changed in the inputs

In [ ]:
topic_psi = categorical_psi([e["topic"] for e in week1], [e["topic"] for e in week4])
# a 1-D proxy for the question-embedding projection
emb_ref = np.random.default_rng(0).normal(0, 1, 3000)
emb_now = np.random.default_rng(0).normal(0.7, 1.2, 1500)
mon = DriftMonitor(emb_ref)
print("topic-label PSI:", round(topic_psi, 3))
print("embedding drift:", mon.assess(emb_now))

## 3 — Trigger: fire only on a real, multi-signal degradation

In [ ]:
trig = RefreshTrigger()
decision = trig.decide(psi_value=topic_psi, signals=s4, eval_score=0.71, baseline_eval=0.83)
print(decision)

## 4 — Rebuild → ReleaseGate → CanaryController

In [ ]:
baseline = EvalReport("bundle-v7", {f"q{i}": 0.83 for i in range(20)})

def rebuild_and_eval():
    # re-ingest so 'new_product' is covered -> per-case scores recover
    return EvalReport("bundle-v8", {f"q{i}": 0.86 for i in range(20)})

def run_refresh(decision):
    if not decision.refresh:
        return "no action"
    candidate = rebuild_and_eval()
    verdict = ReleaseGate(baseline).check(candidate)
    print("  gate:", verdict.notes, "-> ok =", verdict.ok)
    if not verdict.ok:
        return "BLOCKED at eval gate"
    profiles = {"v7": dict(lat=0.9, err=0.01, q=0.82), "v8": dict(lat=1.1, err=0.015, q=0.86)}
    canary = CanaryController(measure_factory(profiles), stable="v7", candidate="v8")
    result = canary.roll_out()
    for step in result.steps:
        print("  canary", step)
    return f"{result.outcome} at {result.final_pct}%"

print(run_refresh(decision))

## 5 — The loop closed

`serve → signals → drift/decay/feedback → trigger → rebuild → gate → canary → promote → new
baseline`. That's the operating loop for an LLM system in production.

**Exercises** (see `README.md` for the full list): add a cooldown to `RefreshTrigger` so a
one-day news spike can't fire it; add a `Shadow` deployment mode to `CanaryController`; make
`ReleaseGate` average the aggregate over N noisy eval runs before blocking.